# MLflow Evaluation Demo: Multi-Turn Restaurant Research Agent with Web Search

![restaurant_food_bot](./images/restaurant_food_bot.png)

This multi-turn conversational agent can be used by **Caspers Kitchens'** clients or customers, caterers, or anyone interested in researching restaurants catered by kitches such as 
**Caspers Kitchens** for the following scenarios:

* **Food allergies** — identify dishes and restaurants that accommodate specific dietary restrictions (e.g. peanut-free, gluten-free, vegan)
* **Restaurant ratings & recommendations** — discover highly rated restaurants by neighborhood, cuisine, or preference
* **Food safety inspections** — look up health inspection scores and recent violation reports for a specific restaurant
* **Menu & hours** — find current operating hours, menus, and vegetarian or allergen-friendly options
* **Personalized recommendations** — get synthesized advice across multiple turns, with the agent remembering your preferences throughout the conversation

This notebook demonstrates MLflow 3.10 evaluation capabilities using a web-search-augmented multi-turn agent:

1. Set up session-level judges for multi-turn conversation evaluation
2. Use `mlflow.genai.evaluate()` to evaluate full conversations
3. Evaluate **three dimensions**: coherence, context retention, and search quality
4. Inspect traces and results in the MLflow UI

## What You'll Learn

- How to create session-level judges with the `{{ conversation }}` template
- How a tool-use loop (web search via Tavily) is traced automatically
- How to evaluate a stateless-search agent for context carryover

---

In [1]:
import mlflow
print(mlflow.__version__)

/Users/jules/git-repos/mlflow-demos/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3.10.0


## Setup: Import Dependencies

In [2]:
import os
import mlflow
import pandas as pd
from pathlib import Path


from devconnect.config import AgentConfig
from devconnect.restaurant_research_bot.restaurant_research_agent_cls import RestaurantResearchAgent
from devconnect.restaurant_research_bot.scenarios import get_scenario_food_safety
from devconnect.restaurant_research_bot.prompts import (
    SYSTEM_PROMPT_NAME,
    COHERENCE_JUDGE_NAME,
    CONTEXT_RETENTION_JUDGE_NAME,
    SEARCH_QUALITY_JUDGE_NAME,
)

# Load environment variables from .env if present
try:
    from dotenv import load_dotenv
    env_file = Path(".env")
    if env_file.exists():
        load_dotenv(env_file)
        print(f"Loaded environment variables from {env_file.absolute()}")
    else:
        print("No .env file found. Set OPENAI_API_KEY and TAVILY_API_KEY manually.")
except ImportError:
    print("python-dotenv not installed. Set environment variables manually.")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

Loaded environment variables from /Users/jules/git-repos/mlflow-demos/devconnect/restaurant_research_bot/.env


## Configuration

In [3]:
# Provider and model configuration
PROVIDER = "openai"         # or "databricks"
AGENT_MODEL = "gpt-5-mini"
JUDGE_MODEL = "gpt-5-mini"
TEMPERATURE = 0.2
EXPERIMENT_NAME = "restaurant-research-bot"

print("Configuration:")
print(f"  Provider:    {PROVIDER}")
print(f"  Agent Model: {AGENT_MODEL}")
print(f"  Judge Model: {JUDGE_MODEL}")
print(f"  Experiment:  {EXPERIMENT_NAME}")

Configuration:
  Provider:    openai
  Agent Model: gpt-5-mini
  Judge Model: gpt-5-mini
  Experiment:  restaurant-research-bot


In [4]:
# Verify required credentials are set
openai_key = os.environ.get("OPENAI_API_KEY", "")
tavily_key = os.environ.get("TAVILY_API_KEY", "")

print(f"OPENAI_API_KEY  set: {'yes' if openai_key else 'NO - required'}")
print(f"TAVILY_API_KEY  set: {'yes' if tavily_key else 'NO - required for web search'}")

if PROVIDER == "databricks":
    db_host = os.environ.get("DATABRICKS_HOST", "")
    db_token = os.environ.get("DATABRICKS_TOKEN", "")
    print(f"DATABRICKS_HOST set: {'yes' if db_host else 'NO - required for databricks'}")
    print(f"DATABRICKS_TOKEN set: {'yes' if db_token else 'NO - required for databricks'}")

OPENAI_API_KEY  set: yes
TAVILY_API_KEY  set: yes


## Step 1: Setup MLflow Tracking

In [5]:
mlflow.openai.autolog()

using_databricks_mlflow = False  # Set to True if you're using Databricks MLflow on Databricks Workspace
if using_databricks_mlflow:
    mlflow.set_tracking_uri("databricks")
    EXPERIMENT_NAME_FULL = f"/Users/jules@databricks.com/{EXPERIMENT_NAME}"
    mlflow.set_experiment(EXPERIMENT_NAME_FULL)
else:
    mlflow.set_tracking_uri("http://localhost:5000")
    mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

2026/03/24 13:48:41 INFO mlflow.tracking.fluent: Experiment with name 'restaurant-research-bot' does not exist. Creating a new experiment.


MLflow tracking URI: http://localhost:5000
Experiment: restaurant-research-bot


## Step 2: Initialize the Restaurant Research Agent

The agent uses a tool-use loop: when it needs current data (hours, ratings, menus),
it calls `web_search()` via Tavily, feeds the results back, and continues.

Each `handle_message()` call produces a `CHAT_MODEL` trace with `TOOL` child spans
for every web search performed.

In [6]:
# Build judge model URI — MLflow make_judge() requires "<provider>:/<model>" format
if PROVIDER == "databricks":
    judge_model_uri = f"databricks:/{JUDGE_MODEL}"
else:
    judge_model_uri = f"openai:/{JUDGE_MODEL}"

config = AgentConfig(
    model=AGENT_MODEL,
    provider=PROVIDER,
    temperature=TEMPERATURE,
    mlflow_experiment=EXPERIMENT_NAME,
)

agent = RestaurantResearchAgent(config=config, judge_model=judge_model_uri, debug=True)

print("\nAgent initialised")
print(f"  Provider: {config.provider}")
print(f"  Model:    {config.model}")
print(f"  Judge:    {judge_model_uri}")

2026/03/24 13:48:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: system_prompt, version 1
2026/03/24 13:48:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: judge_conversation_coherence, version 1
2026/03/24 13:48:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: judge_context_retention, version 1
2026/03/24 13:48:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: judge_search_quality, version 1


  Coherence judge session-level:      True
  Context retention judge session-level: True
  Search quality judge session-level: True

Agent initialised
  Provider: openai
  Model:    gpt-5-mini
  Judge:    openai:/gpt-5-mini


## Step 3: Run a Conversation Scenario

The **Food Safety Research** scenario tests whether the agent:
- Searches appropriately for current inspection data
- Resolves implicit references ("that restaurant" → Nopa) in search queries
- Synthesises findings at the end without re-searching

In [7]:
scenario = get_scenario_food_safety()

print(f"Scenario:   {scenario['name']}")
print(f"Session ID: {scenario['session_id']}")
print(f"Turns:      {len(scenario['messages'])}")
print("\nMessages:")
for i, msg in enumerate(scenario['messages'], 1):
    print(f"  Turn {i}: {msg}")

Scenario:   Food Safety Research
Session ID: session-safety-001
Turns:      4

Messages:
  Turn 1: How can I check food safety inspection scores for restaurants in San Francisco?
  Turn 2: What's the current health inspection rating for Nopa on Divisadero Street?
  Turn 3: Are there any recent violation reports for that restaurant?
  Turn 4: Based on all this, would you say it's safe to eat there?


In [8]:
# Run the conversation inside an MLflow run.
# All handle_message() traces produced here share the run_id,
# which evaluate_session() uses to find them.
with mlflow.start_run(run_name=scenario["name"]) as run:
    agent.run_conversation(scenario["messages"], scenario["session_id"])
    run_id = run.info.run_id

print(f"\nConversation complete. Run ID: {run_id}")


Turn 1/4
  User: How can I check food safety inspection scores for restaurants in San Francisco?
  Bot:  Quick options — official sources first:

1) SF Dept. of Public Health / Food Safety page — search “Get restaurant health inspection results” to look up a restaurant by name or address: https://www.sf.gov/departments/food-safety

2) San Francisco OpenData — interactive dataset of inspection scores (downloadable, filterable): https://data.sfgov.org/ (search for “Health Inspection Scores”)

3) Mobile/third‑party — the “SF Restaurant Health Scores” app and sites like Yelp sometimes show the latest SFDPH score and inspection date.

What you’ll find: a numeric score (higher = safer), inspection date, violation details and inspector notes. Want me to look up a specific restaurant’s latest score? If so, give the name or address.

Turn 2/4
  User: What's the current health inspection rating for Nopa on Divisadero Street?
  Bot:  I can check that for you, but I can't find a single authoritat

[Trace(trace_id=tr-1400b3f569860dddd9cc19f05d57bedf), Trace(trace_id=tr-1e5b2e1adac5ffb71680bc1b5d98ad88), Trace(trace_id=tr-22b5596b8994a4870fec38fd23408623), Trace(trace_id=tr-16fce280018aac898e44837c23aaa784)]

---

# Evaluation Showcase: MLflow Session-Level Judges

We evaluate the conversation along **three dimensions**:

| Judge | Type | What it measures |
|---|---|---|
| `conversation_coherence` | `bool` | Does the conversation flow logically? |
| `context_retention` | `excellent/good/fair/poor` | Does the agent remember prior constraints? |
| `search_quality` | `necessary/unnecessary/skipped` | Did the agent search at the right times? |

## Step 4: View Judge Instructions

All three judges use the `{{ conversation }}` template — this is what makes them
**session-level**: MLflow aggregates all turns and passes the full conversation to each judge.

In [9]:
for name in [
    SYSTEM_PROMPT_NAME,
    COHERENCE_JUDGE_NAME,
    CONTEXT_RETENTION_JUDGE_NAME,
    SEARCH_QUALITY_JUDGE_NAME,
]:
    prompt = mlflow.genai.load_prompt(name)
    print("=" * 70)
    print(f"{name}  (v{prompt.version})")
    print("=" * 70)
    print(prompt.template)
    print()

print("All three judge prompts contain {{ conversation }} -- session-level!")

# Use the prompts in the MLflow UI:
print("View the prompts in the MLflow UI:")
print(f"  http://localhost:5000  ->  '{EXPERIMENT_NAME}' experiment")
print("  Prompts tab -> '{{ conversation }}'")

system_prompt  (v1)
You are a helpful assistant that can search the web to answer questions.

Guidelines:
- Use the web_search tool when you need current information, specific facts,
  business details, menus, hours, ratings, or anything you're not confident about.
- Do NOT search for things you already know well (basic facts, general knowledge).
- Remember everything the user has told you across turns. If they mention a dietary
  restriction in turn 1, apply it automatically in turn 4 without re-asking.
- After searching, synthesize the results into a clear, direct answer.
  Do not dump raw search results at the user.
- Keep responses concise -- under 150 words unless listing multiple items.

IMPORTANT -- search query construction:
The web_search tool has NO memory of prior turns. It is a stateless API that receives
only the query string you pass it. You are responsible for constructing complete,
self-contained queries that incorporate any relevant context from the conversation.

Bad 

## Step 5: Search for Traces with `mlflow.search_traces()`

In [10]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

session_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{scenario['session_id']}'"
)

print(f"Found {len(session_traces)} traces for session '{scenario['session_id']}'")
print("Each trace = one conversation turn (with tool spans for web searches)")

display(session_traces[["request_time", "request", "response"]].sort_values(
    by="request_time", ascending=True
))

Found 4 traces for session 'session-safety-001'
Each trace = one conversation turn (with tool spans for web searches)


,request_time,request,response
3,1774385321477,"{'message': 'How can I check food safety inspection scores for restaurants in San Francisco?', 'session_id': 'session-safety-001'}","Quick options — official sources first:\n\n1) SF Dept. of Public Health / Food Safety page — search “Get restaurant health inspection results” to look up a restaurant by name or address: https://www.sf.gov/departments/food-safety\n\n2) San Francisco OpenData — interactive dataset of inspection scores (downloadable, filterable): https://data.sfgov.org/ (search for “Health Inspection Scores”)\n\n3) Mobile/third‑party — the “SF Restaurant Health Scores” app and sites like Yelp sometimes show the latest SFDPH score and inspection date.\n\nWhat you’ll find: a numeric score (higher = safer), inspection date, violation details and inspector notes. Want me to look up a specific restaurant’s latest score? If so, give the name or address."
2,1774385367875,"{'message': 'What's the current health inspection rating for Nopa on Divisadero Street?', 'session_id': 'session-safety-001'}","I can check that for you, but I can't find a single authoritative current score in public copies — SFDPH’s Health Inspection dataset updates have been paused and third‑party sites show conflicting numbers.\n\nYou can view the official, up‑to‑date report here:\n- SF Dept. of Public Health “Get restaurant health inspection results”: https://www.sf.gov/get-restaurant-health-inspection-results\n- DataSF Health Inspection Scores (official dataset): https://data.sfgov.org/Health-and-Social-Services/Health-Inspection-Scores-2024-Present-/tvy3-wexg\n\nWould you like me to pull the latest SFDPH listing for Nopa (560 Divisadero St) now and report the most recent score and inspection date?"
1,1774385518010,"{'message': 'Are there any recent violation reports for that restaurant?', 'session_id': 'session-safety-001'}","Short answer: I couldn’t find any confirmed, recent violation reports for Nopa (560 Divisadero) in the SFDPH/DataSF records I checked.\n\nNotes and next steps:\n- The City’s DataSF “Health Inspection Scores (2024–Present)” feed has been paused for updates (per DataSF), so public datasets may lag.\n- Third‑party sites (Yelp, local summaries) don’t show a current open violation for Nopa either.\n- For a definitive, up‑to‑the‑minute result you can check SFDPH’s lookup page: https://www.sf.gov/get-restaurant-health-inspection-results\n\nWould you like me to pull the latest official SFDPH record for Nopa now and report the most recent inspection date, score and any violation details?"
0,1774385711519,"{'message': 'Based on all this, would you say it's safe to eat there?', 'session_id': 'session-safety-001'}","Short answer: yes — based on the checks you asked me to run, I didn’t find any confirmed recent violation reports for Nopa (560 Divisadero), so it appears reasonably safe for most patrons.\n\nCaveats and sensible precautions:\n- DataSF’s inspection feed has had update delays, so public records can lag. That means a “no recent violations found” result isn’t an absolute guarantee.\n- Inspections reduce risk but don’t eliminate foodborne illness. If you’re immunocompromised, pregnant, elderly, or serving very young children, be extra cautious (avoid raw/undercooked dishes, shellfish, etc.).\n- Quick on‑site checks help: visible cleanliness, staff hygiene, and how hot/cold foods are held.\n\nWould you like me to pull the latest SFDPH inspection report for Nopa right now (date, score, and any violation details)?"


## Step 6: Evaluate with `mlflow.genai.evaluate()`

`evaluate_session()` internally calls `mlflow.genai.evaluate()` with all three judges.
Each session-level judge receives the **full conversation** via `{{ conversation }}`
rather than evaluating individual turns.

In [11]:
results = agent.evaluate_session(scenario["session_id"], run_id)

print("\nEvaluation complete.")


Evaluating session 'session-safety-001' (4 traces)...

Evaluation complete.


## Step 7: Inspect Results

In [12]:
coh  = results["coherence"]
ctx  = results["context_retention"]
srch = results["search_quality"]

print("=" * 50)
print(f"Session:  {results['session_id']}")
print(f"Traces:   {results['num_traces']}")
print("=" * 50)
print(f"\nCoherence:         {'PASS' if coh['passed'] else 'FAIL'}  ({coh['feedback_value']})")
if coh["rationale"]:
    print(f"  {coh['rationale']}")

print(f"\nContext Retention: {str(ctx['feedback_value']).upper()}")
if ctx["rationale"]:
    print(f"  {ctx['rationale']}")

print(f"\nSearch Quality:    {str(srch['feedback_value']).upper()}")
if srch["rationale"]:
    print(f"  {srch['rationale']}")

Session:  session-safety-001
Traces:   4

Coherence:         PASS  (True)

Context Retention: EXCELLENT

Search Quality:    NECESSARY


In [13]:
# View in MLflow UI
print("View results in MLflow UI:")
print("  mlflow ui")
print(f"  http://localhost:5000  ->  '{EXPERIMENT_NAME}' experiment")
print(f"  Chat Sessions tab -> {scenario['session_id']}")

View results in MLflow UI:
  mlflow ui
  http://localhost:5000  ->  'restaurant-research-bot' experiment
  Chat Sessions tab -> session-safety-001


---

## What Just Happened?

1. **`RestaurantResearchAgent`** ran a 4-turn conversation with a tool-use loop — each
   web search appears as a child `TOOL` span inside the `CHAT_MODEL` trace.

2. **`mlflow.update_current_trace()`** tagged every trace with the session ID, allowing
   MLflow to group all turns as one session.

3. **`make_judge()`** created three session-level judges. The `{{ conversation }}` template
   variable tells MLflow to aggregate all turns before scoring.

4. **`mlflow.genai.evaluate()`** fanned out all traces to the three judges simultaneously.

5. **`search_quality`** is unique to this agent: it verifies the agent searched when it
   needed to (`necessary`), didn't over-search (`unnecessary`), and didn't silently skip
   searches it should have made (`skipped`).